In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_6.py ──
"""
Shared infrastructure for MLFP03 Exercise 6 — Interpretability and Fairness.

Contains: Singapore credit scoring data load, LightGBM model training,
TreeSHAP explainer setup, output directory, and common helper utilities.

Technique-specific code (permutation importance loops, LIME wrappers,
fairness audit reports) lives in the per-technique files under
`modules/mlfp03/solutions/ex_6/`.

Import pattern (solutions and local both):

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input



# ════════════════════════════════════════════════════════════════════════
# PATHS / CONSTANTS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp03_ex6_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Singapore credit scoring: Monetary Authority of Singapore (MAS) requires
# explainability for credit decisions under the Model Risk Management
# guideline. This dataset simulates a retail-bank default prediction task
# used throughout MLFP02/MLFP03.
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"
RANDOM_SEED = 42

# Protected attribute candidates we audit for disparate impact.
PROTECTED_CANDIDATES: list[str] = ["age", "gender", "ethnicity", "marital_status"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOAD + MODEL TRAIN
# ════════════════════════════════════════════════════════════════════════

# Populated on first call so every technique file sees the same split.
_CACHE: dict[str, Any] = {}


def load_credit_scoring() -> dict[str, Any]:
    """Load the Singapore credit scoring dataset and run the M3 preprocessing
    pipeline. Returns a dict with X_train, y_train, X_test, y_test, feature_names.

    The return value is cached so repeated calls from different technique
    files re-use the same split (essential for interpretability comparisons).
    """
    if _CACHE:
        return _CACHE

    loader = MLFPDataLoader()
    credit: pl.DataFrame = loader.load(DATASET_MODULE, DATASET_FILE)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target=TARGET_COLUMN,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_columns = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    feature_names: list[str] = col_info["feature_columns"]

    _CACHE.update(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_names,
    )
    return _CACHE


def train_credit_model() -> dict[str, Any]:
    """Train the LightGBM credit default model. Cached per-process.

    Returns a dict with model, y_proba, y_pred, auc, and all data from
    `load_credit_scoring()`.
    """
    if "model" in _CACHE:
        return _CACHE

    data = load_credit_scoring()
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)

    _CACHE.update(model=model, y_proba=y_proba, y_pred=y_pred, auc=auc)
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# SHAP EXPLAINER
# ════════════════════════════════════════════════════════════════════════


def build_shap_explainer() -> dict[str, Any]:
    """Construct the TreeSHAP explainer and compute SHAP values for X_test.

    Returns the full bundle: model, data, explainer, shap_vals, expected_value.
    """
    if "shap_vals" in _CACHE:
        return _CACHE

    bundle = train_credit_model()
    explainer = shap.TreeExplainer(bundle["model"])
    shap_values = explainer.shap_values(bundle["X_test"])

    # TreeSHAP for binary classifiers may return [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, list)
        else explainer.expected_value
    )

    _CACHE.update(
        explainer=explainer,
        shap_vals=shap_vals,
        expected_value=expected_value,
    )
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# REUSABLE UTILITIES
# ════════════════════════════════════════════════════════════════════════


def rank_features_by_mean_abs_shap(
    shap_vals: np.ndarray, feature_names: list[str]
) -> list[tuple[str, float]]:
    """Return [(feature, mean_abs_shap), ...] sorted descending."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    return sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)


def feature_index(feature_names: list[str], name: str) -> int:
    """Lookup a feature column index by name, raising a clear error."""
    if name not in feature_names:
        raise KeyError(
            f"Feature '{name}' not found. Available: {feature_names[:10]}..."
        )
    return feature_names.index(name)


def synthetic_group_split(
    X: np.ndarray, feature_idx: int = 0
) -> tuple[np.ndarray, np.ndarray, float]:
    """Split X into two groups on a median cut of `feature_idx`.

    Returns (group_a_mask, group_b_mask, median_value).
    Used as a fallback when no protected attribute is present in features.
    """
    vals = X[:, feature_idx]
    median_val = float(np.median(vals))
    group_a = vals <= median_val
    group_b = ~group_a
    return group_a, group_b, median_val


def print_section(title: str, char: str = "=") -> None:
    """Print a standardised section banner."""
    line = char * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(line)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 6.5: Fairness Audit (Disparate Impact, Equalized Odds,
#                                          Impossibility Theorem)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Measure the disparate impact ratio (4/5 rule)
#   - Measure equalized odds (TPR and FPR parity across groups)
#   - Understand the Chouldechova/Kleinberg impossibility theorem
#   - Produce a regulatory-grade fairness audit report
#   - Apply: MAS Fair Dealing Guidelines annual disclosure
#
# PREREQUISITES: 01_shap_global.py
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — multiple incompatible fairness definitions
#   2. Build — group splitting, disparate impact, equalized odds
#   3. Train — AUDIT the trained model
#   4. Visualise — per-group rate table + per-group TPR/FPR
#   5. Apply — MAS Fair Dealing report for SG Retail Bank
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from dotenv import load_dotenv
from sklearn.metrics import confusion_matrix


load_dotenv()



## THEORY — Three Incompatible Fairness Definitions


In [ ]:
#   1. Demographic parity: equal selection rates across groups
#   2. Equalized odds:     equal TPR and FPR across groups
#   3. Calibration:        predicted probs reliable per group
#
# Chouldechova (2017) + Kleinberg et al. (2016): when base rates differ,
# the three are MATHEMATICALLY INCOMPATIBLE. Any production model MUST
# pick one criterion and document the tradeoff.



## TASK 2 — BUILD group-level audit machinery


Compute per-value approval rate and disparate impact ratio.


Compute TPR and FPR for two boolean group masks.


In [ ]:
bundle = build_shap_explainer()
model = bundle["model"]
X_test = bundle["X_test"]
y_test = bundle["y_test"]
y_pred = bundle["y_pred"]
y_proba = bundle["y_proba"]
feature_names: list[str] = bundle["feature_names"]
shap_vals = bundle["shap_vals"]


def disparate_impact_by_attribute(
    X: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
    attr_idx: int,
    attr_name: str,
) -> dict[str, dict]:
    attr_vals = X[:, attr_idx]
    unique_vals = np.unique(attr_vals[~np.isnan(attr_vals)])
    group_rates: dict[str, dict] = {}
    if len(unique_vals) > 10:
        return group_rates
    for val in sorted(unique_vals):
        mask = attr_vals == val
        n_group = int(mask.sum())
        if n_group < 10:
            continue
        # TODO: compute approval_rate = fraction of y_pred[mask] equal to 0
        approval_rate = ____
        default_rate = float(y_proba[mask].mean())
        group_rates[str(val)] = {
            "n": n_group,
            "approval_rate": approval_rate,
            "mean_default_prob": default_rate,
        }
    return group_rates


def equalized_odds_two_groups(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    group_a: np.ndarray,
    group_b: np.ndarray,
) -> dict[str, dict]:
    out: dict[str, dict] = {}
    for name, mask in [("A", group_a), ("B", group_b)]:
        y_g = y_true[mask]
        p_g = y_pred[mask]
        if y_g.sum() > 0 and (y_g == 0).sum() > 0:
            tn, fp, fn, tp = confusion_matrix(y_g, p_g).ravel()
            # TODO: compute TPR = tp / (tp + fn), FPR = fp / (fp + tn)
            tpr = ____
            fpr = ____
        else:
            tpr, fpr = 0.0, 0.0
        out[name] = {"n": int(mask.sum()), "TPR": float(tpr), "FPR": float(fpr)}
    return out



## TASK 3 — AUDIT the trained model


In [ ]:
print_section("Fairness Audit: Disparate Impact by Protected Attribute")

protected_in_model = [f for f in PROTECTED_CANDIDATES if f in feature_names]
di_results: dict[str, dict[str, dict]] = {}

if protected_in_model:
    print(f"Protected attributes in model: {protected_in_model}")
    for attr in protected_in_model:
        attr_idx = feature_index(feature_names, attr)
        rates = disparate_impact_by_attribute(X_test, y_pred, y_proba, attr_idx, attr)
        if len(rates) >= 2:
            di_results[attr] = rates
            majority_val = max(rates, key=lambda v: rates[v]["n"])
            majority_rate = rates[majority_val]["approval_rate"]
            print(f"\n  --- {attr} (majority = {majority_val}) ---")
            print(f"  {'Value':>10} {'N':>6} {'Approval':>10} {'Disp.Impact':>14}")
            print("  " + "─" * 44)
            for val, info in sorted(rates.items()):
                di = info["approval_rate"] / max(majority_rate, 0.001)
                flag = " < 0.8 FAIL" if di < 0.8 else ""
                print(
                    f"  {val:>10} {info['n']:>6} "
                    f"{info['approval_rate']:>10.3f} {di:>14.3f}{flag}"
                )
else:
    print("No protected attributes in feature set; using synthetic split.")
    group_a, group_b, median_val = synthetic_group_split(X_test, feature_idx=0)
    approval_a = float((y_pred[group_a] == 0).mean())
    approval_b = float((y_pred[group_b] == 0).mean())
    di_ratio = approval_a / max(approval_b, 0.001)
    print(f"  Feature: {feature_names[0]} (median split={median_val:.3f})")
    print(f"  Group A: approval={approval_a:.3f} (n={int(group_a.sum())})")
    print(f"  Group B: approval={approval_b:.3f} (n={int(group_b.sum())})")
    print(f"  DIR:     {di_ratio:.3f}  " f"({'PASS' if di_ratio >= 0.8 else 'FAIL'})")



### Equalized odds on synthetic split


In [ ]:
print_section("Fairness Audit: Equalized Odds (synthetic split)", char="─")
group_a, group_b, median_val = synthetic_group_split(X_test, feature_idx=0)
# TODO: call equalized_odds_two_groups with y_test, y_pred, group_a, group_b
eo = ____
print(f"  Group A: n={eo['A']['n']} TPR={eo['A']['TPR']:.3f} FPR={eo['A']['FPR']:.3f}")
print(f"  Group B: n={eo['B']['n']} TPR={eo['B']['TPR']:.3f} FPR={eo['B']['FPR']:.3f}")
tpr_gap = abs(eo["A"]["TPR"] - eo["B"]["TPR"])
fpr_gap = abs(eo["A"]["FPR"] - eo["B"]["FPR"])
print(f"  |TPR_A - TPR_B| = {tpr_gap:.3f}")
print(f"  |FPR_A - FPR_B| = {fpr_gap:.3f}")



### Impossibility theorem


When base rates differ between groups you CANNOT simultaneously satisfy
  demographic parity, equalized odds, and calibration. Pick one.


In [ ]:
print_section("Impossibility Theorem", char="─")
print(
)
base_rate_a = float(y_test[group_a].mean())
base_rate_b = float(y_test[group_b].mean())
print(f"  Base rate A: {base_rate_a:.3f}")
print(f"  Base rate B: {base_rate_b:.3f}")



### SHAP contribution of protected attributes


In [ ]:
importance_ranking = rank_features_by_mean_abs_shap(shap_vals, feature_names)

print_section("SHAP Contribution of Protected Attributes", char="─")
if protected_in_model:
    for attr in protected_in_model:
        attr_idx_audit = feature_index(feature_names, attr)
        attr_shap = shap_vals[:, attr_idx_audit]
        rank = [n for n, _ in importance_ranking].index(attr) + 1
        print(
            f"  {attr}: mean|SHAP|={np.abs(attr_shap).mean():.4f}  "
            f"rank #{rank}/{len(feature_names)}"
        )
else:
    print("  No explicit protected attributes (proxies may still encode them).")



### Checkpoint


In [ ]:
assert eo["A"]["n"] > 0 and eo["B"]["n"] > 0, "Task 3: both groups must be non-empty"
print("\n[ok] Checkpoint — fairness audit complete\n")



## TASK 4 — Print the full audit report


AUDIT SUMMARY:
  1. Disparate impact:   {len(di_results)} protected attrs reviewed
  2. Equalized odds:     |TPR gap|={tpr_gap:.3f}, |FPR gap|={fpr_gap:.3f}
  3. SHAP protected-attribute rank: completed
  4. Impossibility theorem: acknowledged

RECOMMENDATIONS:
  a. Monitor DIR quarterly (threshold 0.8)
  b. Document which fairness criterion is prioritised and why
  c. SHAP explanation for EVERY declined application
  d. Review proxy variable effects (zip, education)


In [ ]:
print_section("FAIRNESS AUDIT REPORT — Credit Default Model")
print(
)



## TASK 5 — APPLY: MAS Fair Dealing Compliance Report


In [ ]:
# MAS 2024 Fair Dealing Guidelines require an annual fairness disclosure
# per algorithmic credit model: DIR table, TPR/FPR parity, written
# justification, escalation path for DIR < 0.8.
#
# BUSINESS IMPACT: One avoided MAS enforcement incident = ~30x payback
# on the S$16,000 annual audit cost.



## DESTINATION-FIRST CLOSE — km.diagnose


In [ ]:
# This lesson built a fairness audit from primitives — disparate impact
# ratio, equal opportunity difference, predictive parity — to internalise
# the regulatory framing. The kailash-ml SDK packages the standard
# diagnostic surface (per-class metrics, severity heuristics, confusion
# matrix) into a single call; group-conditional fairness metrics layer on
# top of that base.
#
# Destination-first: when the journey is internalised, the SDK is one line.

from kailash_ml import diagnose

# `kind="classical_classifier"` dispatches to the sklearn ClassifierMixin
# adapter. The fairness audit's underlying classifier is the bundle's
# `model` already loaded for the audit.
report = diagnose(model, kind="classical_classifier", data=(X_test, y_test), show=False)
print()
print("  km.diagnose model    : audited credit-default classifier")
print(f"  km.diagnose metrics  : {report.metrics}")
print(f"  km.diagnose severity : {report.severity}")
print()
print("km.diagnose: 1 call for the base diagnostic surface; the fairness")
print("metrics above sit on top of it. Destination-first: when the")
print("journey is internalised, the SDK is one line.")



## REFLECTION


[x] Measured disparate impact ratio across protected groups
  [x] Measured TPR and FPR parity (equalized odds)
  [x] Explained the Chouldechova/Kleinberg impossibility theorem
  [x] Audited SHAP contribution of protected attributes
  [x] Produced a regulatory-grade fairness audit report
  [x] Mapped the pipeline to MAS Fair Dealing annual disclosure

  KEY INSIGHT: Fairness is a FAMILY of incompatible definitions. Measure
  all three, document the tradeoff, make the decision legible to auditors.

  END OF EXERCISE 6.


In [ ]:
print_section("WHAT YOU'VE MASTERED")
print(
)

